In [ ]:
%load_ext autoreload
%autoreload 2

import os, json, sys
DIANNE = "/projects/activities/komp-histopath/USERS/domans/DIANNE"
sys.path.insert(0, f"{DIANNE}/dianne-core")
sys.path.insert(0, f"{DIANNE}/dianne-utils")
sys.path.insert(0, f"{DIANNE}/dianne-viewer")
import dianne_core
import dianne_utils as dianne
import dianne_viewer as viewer
dianne.setNotebookWidth(100)
import pandas as pd

projectDir = '/projects/activities/komp-histopath'

classifierPaths = f'{projectDir}/annotations/KOMP-round1/'
if not os.path.exists(classifierPaths):
    os.makedirs(classifierPaths)

all_slide_metadata = pd.read_csv(f'{projectDir}/USERS/domans/dev-komp/combined_cleaned_metadata.tsv', sep='\t', index_col=0).iloc[:]
all_slide_metadata.index = all_slide_metadata.index.str.replace('.ndpi', '')
df_SM_sel = pd.read_csv(f'{projectDir}/USERS/domans/SelectedMiceForExaminationWithMetadata.csv', index_col=0)
sel_slide_metadata = all_slide_metadata.loc[all_slide_metadata['organism_id'].isin(df_SM_sel.index)]

se_slide_to_mouse = sel_slide_metadata['organism_id']
# /projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/DIANNE/scripts/dev/labels/auto-labels.json
with open(f'{projectDir}/USERS/domans/auto-labels.json', 'rb') as tf:
    labels = json.loads(tf.read())

def getTissues(identity, is_control=False):
    val = 'na' if is_control else identity
    _mice = df_SM_sel[df_SM_sel['tissue']==val].index
    _slides = se_slide_to_mouse[se_slide_to_mouse.isin(_mice)].index
    _subset = [s for s in labels.keys() if (identity in labels[s]) and (s.split('.oid')[0] in _slides)]
    return _subset

tissue_types = ['pancreas', 'kidney'][:]
conditions = [(False, 'abnormal'), (True, 'control')]

tissues = {(tissue, label): getTissues(tissue, is_control=is_control)
    for tissue in tissue_types for is_control, label in conditions}
samples = [s for group in tissues.values() for s in group]

dataPath = f'{projectDir}/results-STQ-KOMP-all/'

used_slides = set([s.split('.oid')[0] for s in samples])
dict_sel_slide_metadata = {k.split('.ndpi')[0]: v for k, v in all_slide_metadata.reindex(used_slides).fillna('NA').T.to_dict().items()}
sample_metadata = {s: dict_sel_slide_metadata[sl] for s in samples if (sl:=s.split('.oid')[0]) in dict_sel_slide_metadata.keys()}

for (tissue, label), group in tissues.items():
    for s in group:
        sample_metadata[s]['tissue'] = tissue
        sample_metadata[s]['label'] = label

samples = ['KOMP_C9066_3.oid4','KOMP_A31269_3.oid2', 'J69485_3.oid2', 'X43538_3.oid1',][:]
viewer.viewSTQkomp(dataPath, samples[:], load_features=True, F=1, classifierPaths=classifierPaths, body_overlap=0.01,
                   save_path=classifierPaths, sample_metadata=sample_metadata, mpp=0.5, username='auto');